# Multi-Modal LLM Style Analysis: run it in Google Colab

This notebook runs our Phase B research pipeline (Braude College capstone project 26-2-R-6) in the
browser. Nothing is installed on your own computer.

**To run it:** Runtime → Run all. When Colab asks for access to Google Drive, allow it.

| Part | What it shows | Needs a Gemini key? |
|---|---|---|
| 1. Setup | Downloads our code from GitHub and installs it | No |
| 2. The corpus | The 49 creations and the integrity check | No |
| 3. The whole method on a tiny example | All six steps, end to end, with a stand-in model | No |
| 4. The method on real text | Gemini rewrites passages of *Macbeth* and *Henry VIII*; each step's output | Yes |
| 5. Run it again | The same numbers from saved answers, with no new calls | Yes |

**The Gemini key** (Parts 4 and 5): click the key icon on the left (Secrets) → Add new secret →
name `GEMINI_API_KEY`, paste the key, and switch on *Notebook access*. Without a key, Parts 4 and 5
are skipped and everything else still runs.

Results are saved to your Google Drive, in the folder `MyDrive/mmlsa`, so nothing is lost when Colab
disconnects.

## Part 1. Setup

The settings below are the only things you may want to change. Part 4 rewrites `CHUNKS` passages and
makes one profile call, so it uses `CHUNKS + 1` calls of the free tier's 20 a day.

In [ ]:
MODEL = "gemini-3.1-flash-lite"  # the model we recommend for the runs (docs/DECISIONS.md I28)
PLAYS = ["the_tragedy_of_macbeth", "king_henry_viii"]  # one expected authentic, one co-written
CHUNKS = 6  # passages Gemini rewrites in Part 4
FREE_TIER = True  # paces the calls to the free tier's limits; set to False with a paid key
REPO_URL = "https://github.com/eliranmeleh/Multi-Modal-LLM-Style-Analysis.git"

Download the code and install it. In Colab this takes about a minute. On a laptop, run the notebook
from inside the repository and this step only finds the code.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def run_quietly(command):
    # Run a setup command and show its output only if it fails.
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        print(result.stdout, result.stderr)
        raise RuntimeError("setup step failed: " + " ".join(command))


if IN_COLAB:
    REPO_DIR = Path("/content/Multi-Modal-LLM-Style-Analysis")
    if not REPO_DIR.exists():
        run_quietly(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    run_quietly([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[gemini]"])
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))  # so this notebook can import our code directly
importlib.invalidate_caches()
print("Our code is ready in", REPO_DIR)

Connect Google Drive, where the results are kept, and read the Gemini key from Colab Secrets.

In [ ]:
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/mmlsa")
    try:
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:  # no such secret, or Notebook access is switched off
        print("No GEMINI_API_KEY in Colab Secrets, so Parts 4 and 5 will be skipped.")
else:
    SAVE_DIR = Path("runs/notebook")  # inside the repository, ignored by git

SAVE_DIR.mkdir(parents=True, exist_ok=True)
HAVE_KEY = bool(os.environ.get("GEMINI_API_KEY")) or (not IN_COLAB and Path(".env").exists())
print("Results are saved in", SAVE_DIR.resolve())
print("Gemini key:", "found" if HAVE_KEY else "not found")

Every step below runs our command-line tool, `python -m mmlsa`, exactly as on a laptop. This small
helper runs it, prints what it says, and ends with how many model answers were new and how many came
from saved copies.

In [ ]:
import json


def mmlsa(*args):
    command = [sys.executable, "-m", "mmlsa", *args]
    environment = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    calls = None
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=environment,
    ) as process:
        for line in process.stdout:
            try:
                event = json.loads(line) if line.startswith("{") else None
            except json.JSONDecodeError:
                event = None
            routine = line.startswith("AFC is enabled") or (
                line.startswith("HTTP Request:") and "200 OK" in line
            )
            if event is None and not routine:
                print(line, end="")  # what the tool says to people, and any error from Gemini
            elif event is not None and event.get("event") == "runner.complete":
                calls = event  # its detailed log is machine-readable; keep only the totals
    if calls:
        print(f"\nModel answers: {calls['live']} new, {calls['cached']} from saved copies")
    if process.returncode != 0:
        raise RuntimeError(f"mmlsa {args[0]} stopped with exit code {process.returncode}")

## Part 2. The corpus

The input is the 49 creations attributed to Shakespeare, taken from Project Gutenberg and cleaned
(headers, licences and cast lists removed). The table lists them by length. The check after it
recomputes every file's fingerprint and confirms that nothing changed since we built the corpus.

In [ ]:
import pandas as pd
from IPython.display import display

manifest = json.loads(Path("data/manifest.json").read_text(encoding="utf-8"))
corpus = (
    pd.DataFrame(manifest["sets"]["texts"])[["title", "n_words"]]
    .rename(columns={"title": "creation", "n_words": "words"})
    .sort_values("words", ascending=False)
    .reset_index(drop=True)
)
print(f"{len(corpus)} creations, {corpus['words'].sum():,} words")
display(corpus)

In [ ]:
mmlsa("corpus", "verify")

## Part 3. The whole method on a tiny example

Three short texts and a **stand-in model** that edits text by fixed rules instead of a real AI. The
numbers mean nothing; the point is to watch all six steps run in seconds and to see what the final
output looks like: one score per text, a threshold, and the chart we will draw for all 49 creations.

In [ ]:
mmlsa(
    "run",
    "--config",
    "configs/mini.yaml",
    "--set",
    f"run.out_dir={(SAVE_DIR / 'runs').as_posix()}",
    "--set",
    f"llm.cache_dir={(SAVE_DIR / 'cache').as_posix()}",
)
mini_run = max((SAVE_DIR / "runs").glob("*-mini"), key=lambda path: path.stat().st_mtime)

**Steps 5 and 6.** `score_mean` is each text's average style distance over its passages and runs
(Step 5). Otsu's method then places the threshold `tau` where it best splits the scores into two
groups (Step 6). Texts above it are labelled *suspicious*.

In [ ]:
from IPython.display import Image

scores = pd.read_csv(mini_run / "scores.csv")
display(scores[["title", "n_chunks", "score_mean", "score_std", "label"]])
threshold = json.loads((mini_run / "threshold.json").read_text(encoding="utf-8"))
print(f"Threshold tau = {threshold['tau']:.4f} ({threshold['method']})")
print("Suspicious:", ", ".join(threshold["suspicious"]) or "none")
display(Image(filename=str(mini_run / "figures" / "sorted_scatter.png")))

## Part 4. The method on real text, step by step

Gemini reads *Macbeth* and *Henry VIII* and writes a style profile (Step 1). The plays are cut into
400-word passages (Step 2), and Gemini rewrites a few of them in the profile's style (Step 3). Then we
measure how much each rewrite changed (Step 4). This part needs the Gemini key. It takes a few
minutes, because the free tier allows four calls and 250,000 input tokens a minute, and the profile
call alone sends both plays, about 70,000 tokens. Lines that say `503` or `429` mean Gemini was busy;
the tool waits and tries again by itself.

In [ ]:
import time

COMPARE = [
    "compare",
    "--config",
    "configs/default.yaml",
    "--model",
    f"gemini:{MODEL}",
    "--chunks",
    str(CHUNKS),
    "--set",
    f"corpus.include_ids=[{','.join(PLAYS)}]",
    "--set",
    f"llm.cache_dir={(SAVE_DIR / 'cache').as_posix()}",
    "--yes",
]
if FREE_TIER:
    # Our rate limiter estimates tokens from words, which undercounts by about a third, so the
    # budget is set well under the real 250,000; retries wait longer when Gemini is busy (503).
    COMPARE += ["--set", "llm.tokens_per_minute=100000", "--set", "llm.max_transient_retries=8"]

live_dir = SAVE_DIR / "live" / time.strftime("%Y%m%d-%H%M%S")
if HAVE_KEY:
    mmlsa(*COMPARE, "--out-dir", live_dir.as_posix())
else:
    print("Skipped: no Gemini key. Add GEMINI_API_KEY in Colab Secrets and run this cell again.")

LIVE_OK = any((live_dir / "profiles").glob("*.json"))
if HAVE_KEY and not LIVE_OK:
    print("\nGemini did not complete the profile this time; the free tier is sometimes busy.")
    print("Wait a minute and run this cell again. The cells below are skipped until it works.")

### Step 1. The style profile

What Gemini says the common style of the two plays is. The prompt never names the author.

In [ ]:
import textwrap

if LIVE_OK:
    profile_file = next((live_dir / "profiles").glob("*.json"))
    profile = json.loads(profile_file.read_text(encoding="utf-8"))["profile"]
    for name, text in (profile.get("fields") or {"profile": profile.get("text", "")}).items():
        print(name.replace("_", " ").upper())
        print(textwrap.fill(str(text), 100), end="\n\n")

### Steps 2 and 3. A passage and its rewrite

Step 2 cut the plays into 400-word passages; on the left is one of them. On the right is Gemini's
rewrite of it (Step 3). Highlighted are the **function words** (*the, of, and, thou, you, ...*) that
the rewrite changed: yellow in the original, green in the rewrite. Change `PASSAGE` to see another one.

In [ ]:
import html
import re

import Levenshtein
from IPython.display import HTML

from mmlsa.distance.fwed import FunctionWordEditDistance, encode_sequence
from mmlsa.distance.tokenize import (
    EDGE_PUNCTUATION,
    extract_function_words,
    load_function_words,
    normalize_for_tokenizing,
)

PASSAGE = 0  # 0 to CHUNKS - 1
FUNCTION_WORDS_FILE = "data/function_words/en_core_v1.txt"
FUNCTION_WORDS = load_function_words(FUNCTION_WORDS_FILE)
FUNCTION_WORD_SET = frozenset(FUNCTION_WORDS)


def read_passage(path):
    # A side-by-side file holds the original passage, then the model's cleaned rewrite.
    sections = re.split(r"^## ", path.read_text(encoding="utf-8"), flags=re.MULTILINE)
    original = sections[1].removeprefix("original").strip()
    header, _, rewrite = sections[2].partition("\n")
    return original, rewrite.strip(), header.strip()


def changed_positions(original, rewrite):
    # The positions FWED counts as changed, from the same alignment the metric uses.
    before = extract_function_words(original, FUNCTION_WORDS)
    after = extract_function_words(rewrite, FUNCTION_WORDS)
    operations = Levenshtein.editops(
        encode_sequence(before, FUNCTION_WORDS), encode_sequence(after, FUNCTION_WORDS)
    )
    removed = {source for kind, source, _ in operations if kind in ("replace", "delete")}
    added = {target for kind, _, target in operations if kind in ("replace", "insert")}
    return removed, added


def marked(text, positions, colour):
    # The passage word by word, with the function words at the given positions highlighted.
    words, index = [], 0
    for word in normalize_for_tokenizing(text).split():
        shown = html.escape(word)
        if word.casefold().strip(EDGE_PUNCTUATION) in FUNCTION_WORD_SET:
            if index in positions:
                shown = f'<mark style="background:{colour};color:black">{shown}</mark>'
            index += 1
        words.append(shown)
    return " ".join(words)


if LIVE_OK:
    comparison = pd.read_csv(live_dir / "comparison.csv")
    chosen = comparison[comparison["ok"]].iloc[PASSAGE]  # only rewrites that passed validation
    passage_file = live_dir / "rewrites" / f"{chosen.creation_id}_{chosen.chunk_index:04d}.md"
    original, rewrite, header = read_passage(passage_file)
    removed, added = changed_positions(original, rewrite)
    cell = "vertical-align:top;text-align:left;line-height:1.5"
    display(
        HTML(
            f"<p><b>{passage_file.stem}</b>: {html.escape(header)}</p>"
            "<table style='width:100%;table-layout:fixed'><tr>"
            f"<th style='{cell}'>Original passage (Step 2)</th>"
            f"<th style='{cell}'>Gemini's rewrite (Step 3)</th></tr>"
            f"<tr><td style='{cell}'>{marked(original, removed, '#ffe08a')}</td>"
            f"<td style='{cell}'>{marked(rewrite, added, '#b6f0b6')}</td></tr></table>"
        )
    )

### Step 4. The style distance (FWED)

Take the function words of each passage, in order. FWED is the number of edits needed to turn one
sequence into the other, divided by the length of the longer sequence. 0 means no change; a passage
already in the corpus style should score low.

In [ ]:
if LIVE_OK:
    result = FunctionWordEditDistance(FUNCTION_WORDS_FILE, with_detail=True)(original, rewrite)
    edits = result.detail["edits"]
    longer = max(result.n_units_original, result.n_units_rewrite)
    print(
        f"Function words: {result.n_units_original} in the original, "
        f"{result.n_units_rewrite} in the rewrite"
    )
    for edit in edits:
        if edit["op"] == "replace":
            print(f"  replace '{edit['from']}' with '{edit['to']}'")
        elif edit["op"] == "delete":
            print(f"  remove  '{edit['from']}'")
        else:
            print(f"  add     '{edit['to']}'")
    print(f"FWED = {len(edits)} edits / {longer} = {result.value:.4f}")
    print(f"The pipeline recorded {chosen.delta:.4f} for this passage in Part 4: the same number.")

### Steps 5 and 6, as a preview

Step 5 averages each play's passages. Step 6, the threshold, needs the whole corpus, so on two plays
and a handful of passages this is only a preview of the shape of the result; Part 3 showed the real
mechanism, and the full run (experiment E4) uses every passage of all 49 creations.

In [ ]:
if LIVE_OK:
    usable = comparison[comparison["ok"]]
    print(f"{len(usable)} of {len(comparison)} rewrites passed the validation checks")
    per_play = usable.groupby("creation_id")["delta"].agg(["count", "mean"])
    display(per_play.rename(columns={"count": "passages", "mean": "mean FWED"}))

## Part 5. Run it again

Every answer Gemini gave was saved. Running the same command again asks Gemini nothing: the answers
come from the saved copies, and the numbers are identical. This is how the peer examiner will
reproduce our results, without a key and without cost.

In [ ]:
if LIVE_OK:
    replay_dir = SAVE_DIR / "replay" / time.strftime("%Y%m%d-%H%M%S")
    mmlsa(*COMPARE, "--out-dir", replay_dir.as_posix())
    first = pd.read_csv(live_dir / "comparison.csv")["delta"]
    again = pd.read_csv(replay_dir / "comparison.csv")["delta"]
    print("Same numbers as the first run:", first.equals(again))

## Where the results are

In Colab they are in Google Drive, under **My Drive → mmlsa**: `runs` (Part 3), `live` and `replay`
(Parts 4 and 5), and `cache`, the saved model answers that make Part 5 free.

In [ ]:
for folder in sorted(SAVE_DIR.iterdir()):
    print(folder.name)